# 🏒 Hockey Board Segmentation — Training on Colab

This notebook fine-tunes a U-Net to segment the dasher board ad zone in NHL broadcast footage.

## Setup steps
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough)
2. Upload the zip from your local machine: `python3 prepare_colab_upload.py` → uploads `colab_training_data.zip`
3. Run all cells
4. Download the output model and replace `src/calibration/board_segmentation_model.pth`

## 1 — Upload training data

In [ ]:
from google.colab import files
import os

print('Upload colab_training_data.zip (generated by prepare_colab_upload.py on your Mac)')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
import zipfile
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# List what we have
for root, dirs, files_ in os.walk('annotation_frames'):
    for f in sorted(files_):
        print(os.path.join(root, f))

## 2 — Install dependencies

In [ ]:
!pip install -q opencv-python-headless scipy

## 3 — Model definition

In [ ]:
import torch
import torch.nn as nn

TARGET_WIDTH  = 640
TARGET_HEIGHT = 360

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._conv_block(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._conv_block(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = self._conv_block(128, 256)
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(256, 128)
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(128, 64)
        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(64, 32)
        self.final = nn.Conv2d(32, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.upconv3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        return self.sigmoid(self.final(d1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model = UNet().to(device)

# Load pre-trained weights if present
import os
if os.path.exists('board_segmentation_model.pth'):
    model.load_state_dict(torch.load('board_segmentation_model.pth', map_location=device))
    print('Loaded pre-trained weights — fine-tuning')
else:
    print('Training from scratch')

## 4 — Ground-truth mask generation from annotations

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import median_filter

RED_LO1 = np.array([0,   150, 150], np.uint8);  RED_HI1 = np.array([10,  255, 255], np.uint8)
RED_LO2 = np.array([170, 150, 150], np.uint8);  RED_HI2 = np.array([180, 255, 255], np.uint8)
YEL_LO  = np.array([18,  150, 150], np.uint8);  YEL_HI  = np.array([35,  255, 255], np.uint8)

def extract_mask_from_annotation(ann_bgr, orig_bgr):
    h, w = ann_bgr.shape[:2]
    hsv = cv2.cvtColor(ann_bgr, cv2.COLOR_BGR2HSV)

    red = cv2.bitwise_or(
        cv2.inRange(hsv, RED_LO1, RED_HI1),
        cv2.inRange(hsv, RED_LO2, RED_HI2))
    red = cv2.morphologyEx(red, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_RECT, (31, 3)))
    yel = cv2.inRange(hsv, YEL_LO, YEL_HI)
    yel = cv2.morphologyEx(yel, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_RECT, (31, 3)))

    top_y = np.full(w, np.nan, dtype=np.float32)
    bot_y = np.full(w, np.nan, dtype=np.float32)

    for col in range(w):
        r_ys = np.where(red[:, col] > 0)[0]
        y_ys = np.where(yel[:, col] > 0)[0]
        if r_ys.size: top_y[col] = float(r_ys.min())
        if y_ys.size: bot_y[col] = float(y_ys.max())

    valid = ~(np.isnan(top_y) | np.isnan(bot_y))
    if valid.sum() < w * 0.1:
        return None

    # Crop to annotated column range
    lo, hi = np.where(valid)[0][[0, -1]]
    top_y[:lo] = np.nan; top_y[hi+1:] = np.nan
    bot_y[:lo] = np.nan; bot_y[hi+1:] = np.nan

    # Interpolate + smooth
    def interp_nans(a):
        mask = np.isnan(a)
        if mask.all(): return a
        xs = np.arange(len(a))
        a[mask] = np.interp(xs[mask], xs[~mask], a[~mask])
        return a

    sw = max(11, w // 12) | 1
    top_y = median_filter(interp_nans(top_y), size=sw).astype(np.int32)
    bot_y = median_filter(interp_nans(bot_y), size=sw).astype(np.int32)
    top_y = np.clip(top_y, 0, h-1)
    bot_y = np.clip(bot_y, 0, h-1)

    mask = np.zeros((h, w), dtype=np.uint8)
    rows = np.arange(h)[:, None]
    mask[(rows >= top_y[None,:]) & (rows <= bot_y[None,:])] = 255

    # Cleanup
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_RECT, (17, 5)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                             cv2.getStructuringElement(cv2.MORPH_RECT, (5, 3)))
    mask[:int(h*0.25), :int(w*0.35)] = 0  # scorebug
    return mask


FRAME_PAIRS = [
    ('v1_f010_overhead_end_zone.jpg', 'v1_f010_overhead_end_zone.png'),
    ('v1_f317_overhead_end_zone.jpg', 'v1_f317_overhead_end_zone.png'),
    ('v1_f634_overhead_end_zone.jpg', 'v1_f634_overhead_end_zone.png'),
    ('v3_f010_overhead_full.jpg',     'v3_f010_overhead_full.png'),
    ('v3_f200_overhead_full.jpg',     'v3_f200_overhead_full.png'),
    ('v4_f010_overhead_near.jpg',     'v4_f010_overhead_near.png'),
]

ANN_DIR = 'annotation_frames'
pairs   = []

for orig_name, ann_name in FRAME_PAIRS:
    orig = cv2.imread(f'{ANN_DIR}/{orig_name}')
    ann  = cv2.imread(f'{ANN_DIR}/{ann_name}')
    if orig is None or ann is None:
        print(f'  SKIP {orig_name} (missing file)')
        continue
    mask = extract_mask_from_annotation(ann, orig)
    if mask is None:
        print(f'  SKIP {orig_name} (no annotation lines found)')
        continue
    pairs.append((orig, mask))
    print(f'  OK {orig_name}: {(mask>0).mean():.1%} board')

print(f'\n{len(pairs)} frames loaded')

## 5 — Visualize ground truth (sanity check)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (orig, mask) in zip(axes.flat, pairs):
    vis = orig.copy()
    vis[mask > 0] = (vis[mask > 0] * 0.4 + np.array([0, 200, 0]) * 0.6).astype(np.uint8)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Board: {(mask>0).mean():.1%}')
    ax.axis('off')
plt.suptitle('Ground Truth Masks (GREEN = board zone)', fontsize=14)
plt.tight_layout()
plt.show()

## 6 — Dataset & Training

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

EPOCHS         = 150
LR             = 5e-4
AUGMENT_FACTOR = 20
BATCH_SIZE     = 8

class BoardDataset(Dataset):
    def __init__(self, pairs, augment=True):
        self.items = []
        for orig, mask in pairs:
            self.items.append((orig, mask))
            if augment:
                for _ in range(AUGMENT_FACTOR - 1):
                    self.items.append(self._aug(orig, mask))

    @staticmethod
    def _aug(img, mask):
        h, w = img.shape[:2]
        # Horizontal flip
        if np.random.rand() > 0.5:
            img = cv2.flip(img, 1); mask = cv2.flip(mask, 1)
        # Brightness/contrast
        a = np.random.uniform(0.70, 1.30); b = int(np.random.uniform(-30, 30))
        img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)
        # HSV jitter
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int16)
        hsv[...,0] = (hsv[...,0] + np.random.randint(-10, 10)) % 180
        hsv[...,1] = np.clip(hsv[...,1] + np.random.randint(-25, 25), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        # Random crop
        s = np.random.uniform(0.85, 1.0)
        nh, nw = int(h*s), int(w*s)
        y0 = np.random.randint(0, h-nh+1); x0 = np.random.randint(0, w-nw+1)
        img  = cv2.resize(img [y0:y0+nh, x0:x0+nw], (w, h))
        mask = cv2.resize(mask[y0:y0+nh, x0:x0+nw], (w, h),
                           interpolation=cv2.INTER_NEAREST)
        # Gaussian blur (simulates camera defocus)
        if np.random.rand() > 0.6:
            k = np.random.choice([3, 5])
            img = cv2.GaussianBlur(img, (k, k), 0)
        return img, mask

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img, mask = self.items[idx]
        img  = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
                           (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (TARGET_WIDTH, TARGET_HEIGHT),
                           interpolation=cv2.INTER_NEAREST)
        x = torch.from_numpy(img).float().permute(2,0,1) / 255.0
        y = torch.from_numpy((mask > 0).astype(np.float32)).unsqueeze(0)
        return x, y

dataset    = BoardDataset(pairs, augment=True)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
print(f'Dataset: {len(dataset)} samples ({len(pairs)} frames × {AUGMENT_FACTOR})')

# Logits model for training (skip sigmoid for BCEWithLogitsLoss)
class UNetLogits(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x):
        e1 = self.base.enc1(x)
        e2 = self.base.enc2(self.base.pool1(e1))
        e3 = self.base.enc3(self.base.pool2(e2))
        b  = self.base.bottleneck(self.base.pool3(e3))
        d3 = self.base.dec3(torch.cat([self.base.upconv3(b), e3], dim=1))
        d2 = self.base.dec2(torch.cat([self.base.upconv2(d3), e2], dim=1))
        d1 = self.base.dec1(torch.cat([self.base.upconv1(d2), e1], dim=1))
        return self.base.final(d1)

model_logits = UNetLogits(model).to(device)
pos_weight   = torch.tensor([3.0]).to(device)   # penalise false negatives more
criterion    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer    = optim.AdamW(model_logits.parameters(), lr=LR, weight_decay=1e-4)
scheduler    = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.02)

losses = []
best_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model_logits.train()
    ep_loss = 0.0
    for x_b, y_b in dataloader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model_logits(x_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_logits.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    avg = ep_loss / len(dataloader)
    losses.append(avg)
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), 'board_segmentation_model.pth')
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg:.4f}  best={best_loss:.4f}')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')

## 7 — Loss curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Training Loss'); plt.grid(True)
plt.tight_layout(); plt.show()

## 8 — Validate predictions vs ground truth

In [ ]:
model.eval()
fig, axes = plt.subplots(len(pairs), 3, figsize=(18, 4 * len(pairs)))
if len(pairs) == 1: axes = [axes]

threshold = 0.18

for i, (orig, gt_mask) in enumerate(pairs):
    h, w = orig.shape[:2]
    img = cv2.resize(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB),
                      (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    x = torch.from_numpy(img).float().permute(2,0,1).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        prob = model(x).squeeze().cpu().numpy()
    prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    pred_mask = (prob > threshold).astype(np.uint8) * 255

    # Compute IoU
    gt_b   = gt_mask   > 0
    pred_b = pred_mask > 0
    inter  = (gt_b & pred_b).sum()
    union  = (gt_b | pred_b).sum()
    iou    = inter / union if union > 0 else 0.0

    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)

    def overlay(base, mask, color):
        vis = base.copy()
        vis[mask > 0] = (vis[mask > 0] * 0.4 + np.array(color) * 0.6).astype(np.uint8)
        return vis

    axes[i][0].imshow(orig_rgb); axes[i][0].set_title('Original'); axes[i][0].axis('off')
    axes[i][1].imshow(overlay(orig_rgb, gt_mask,   [0, 200, 0]))
    axes[i][1].set_title('Ground Truth'); axes[i][1].axis('off')
    axes[i][2].imshow(overlay(orig_rgb, pred_mask, [0, 120, 255]))
    axes[i][2].set_title(f'Predicted  IoU={iou:.2%}'); axes[i][2].axis('off')

plt.suptitle('GT (green) vs Predicted (blue)', fontsize=14)
plt.tight_layout(); plt.show()

## 9 — Download the trained model

In [ ]:
from google.colab import files
files.download('board_segmentation_model.pth')
print('Downloaded! Replace src/calibration/board_segmentation_model.pth in your repo.')